# BEVFormer V2 — multi-scale lifting, step by step
### _How adding a feature pyramid turns a single-scale BEV encoder into a multi-object detector_

| | |
|---|---|
| **Lifter key** | `bevformer` with `num_levels=2` |
| **V1 paper** | Li et al., _BEVFormer_, ECCV 2022 · arXiv:2203.17270 |
| **Simple-BEV** | Harley et al., IROS 2023 · `bevformernet2.py` |
| **Deformable DETR** | Zhu et al., ICLR 2021 · arXiv:2010.04159 |
| **FPN** | Lin et al., CVPR 2017 |

**This notebook covers (new material only — read `bevformer_lifting_step_by_step.ipynb` first):**

1. Why a feature pyramid matters and how it is built
2. `flatten_pyramid` — line-by-line code trace
3. `level_embed` — the math, weight shapes, PCA ablation
4. `PillarDeformableAttention` internals — offset network, sampling locations, aggregation
5. `SpatialCrossAttention` — per-camera attention and visibility-masked fusion
6. BEV query evolution across encoder layers
7. Training comparison V1 vs V2 + loss curves + mIoU
8. Ablations: no `level_embed`, varying `num_levels`
9. Geometry verification (90° rig rotation)
10. Camera dropout and error analysis


## Prerequisites

> Read `bevformer_lifting_step_by_step.ipynb` first.  
> That notebook explains BEV queries, pillar anchors, and single-scale deformable cross-attention.  
> Here we focus **only on what changes** when you add a second pyramid level.


In [1]:
import math, time, copy, dataclasses
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.facecolor':'#1a1a2e','axes.facecolor':'#16213e',
                     'axes.edgecolor':'#e0e0e0','text.color':'#e0e0e0',
                     'axes.labelcolor':'#e0e0e0','xtick.color':'#e0e0e0',
                     'ytick.color':'#e0e0e0','grid.color':'#334466',
                     'grid.linestyle':'--','grid.alpha':0.5})

import sys, os
for _p in [os.path.join('..','src'),'src']:
    if os.path.isdir(os.path.join(_p,'lifting')): sys.path.insert(0,_p); break

from lifting.geometry import GridSpec
from lifting.data import SyntheticSceneDataset, SceneBatch
from lifting.lifters.bevformer import BEVFormerLifter
from lifting.lifters.bevformer.bevformer_layer import BEVFormerLayer
from lifting.lifters.pyramid import flatten_pyramid
from lifting.attention import SpatialCrossAttention, PillarDeformableAttention
from lifting.models import BEVSegmentationModel
from lifting.decoders import BEVDecoder, SegmentationHead
from lifting.training import Trainer, BEVSegmentationTask
from lifting.encoders import SimpleConvEncoder
from lifting.metrics import IoUMetric


def make_bev_model(encoder, lifter, num_classes):
    """Encoder -> lifter -> BEV decoder -> segmentation head (the same assembly as build_bev_model)."""
    channels = encoder.out_channels
    return BEVSegmentationModel(encoder, lifter, BEVDecoder(channels, channels), SegmentationHead(channels, num_classes))


def make_trainer(model, num_classes, lr=3e-4):
    """AdamW trainer with the BEV segmentation loss."""
    return Trainer(model, BEVSegmentationTask(num_classes), lr=lr)

print('imports ok')


imports ok


In [2]:
torch.manual_seed(42)
grid = GridSpec(resolution=(32,32,4))
data = SyntheticSceneDataset(1024, grid, seed=0)  # 600 training scenes + held-out test scenes
NUM_CLASSES = 1 + len(data.sampler.classes)  # background + every object class the sampler draws
batch = SceneBatch.collate([data[i] for i in range(4)])
B_b, N_b = batch.images.shape[:2]
H_img = batch.images.shape[-2]
print(f'BEV grid : {grid.bev_shape}  (X x Y cells)')
print(f'Batch    : images {tuple(batch.images.shape)}  labels {tuple(batch.bev_labels.shape)}')


BEV grid : (32, 32)  (X x Y cells)
Batch    : images (4, 4, 3, 48, 96)  labels (4, 32, 32)


In [3]:
import importlib.util, pathlib
_found = False
for _p in ['.','notebooks','../notebooks']:
    _f = pathlib.Path(_p)/'lifting_diagrams.py'
    if _f.exists():
        spec = importlib.util.spec_from_file_location('dg',str(_f))
        dg = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(dg)
        _found = True; break
assert _found, 'lifting_diagrams.py not found'
print('lifting_diagrams loaded')


lifting_diagrams loaded


---
## Part I — Feature Pyramid Network
### Step 1 · Why multiple scales?

A backbone at stride 32 gives ~4 pixels for a pedestrian at 30 m — invisible.
FPN taps the backbone at every stage:

```
Level  stride  spatial   receptive    good for
  0      x8    H/8  W/8    ~64 px    pedestrians, signs
  1     x16   H/16 W/16   ~128 px    cars, lane markings
  2     x32   H/32 W/32   ~256 px    trucks, junctions
```

BEVFormer V2 reads levels {0, 1} simultaneously so the same BEV query sees both fine texture and coarse structure.

$$\mathcal{F} = \{F_l \in \mathbb{R}^{B \times N \times C \times H_l \times W_l}\}_{l=0}^{L-1}$$

where $H_l = H_0 / 2^l$, $W_l = W_0 / 2^l$.


In [4]:
C = 32
enc1 = SimpleConvEncoder(out_channels=C, num_levels=1)
enc2 = SimpleConvEncoder(out_channels=C, num_levels=2)
enc3 = SimpleConvEncoder(out_channels=C, num_levels=3)

with torch.no_grad():
    imgs_flat = batch.images.flatten(0,1)
    pyr1 = enc1(imgs_flat)
    pyr2 = enc2(imgs_flat)
    pyr3 = enc3(imgs_flat)

pyr2_bn = [f.view(B_b,N_b,*f.shape[1:]) for f in pyr2]

print('2-level pyramid (V2 default):')
for l,f in enumerate(pyr2):
    print(f'  level {l}: {tuple(f.shape)}  stride x{H_img//f.shape[-2]}')
print('\n3-level pyramid (V3):')
for l,f in enumerate(pyr3):
    print(f'  level {l}: {tuple(f.shape)}  stride x{H_img//f.shape[-2]}')


2-level pyramid (V2 default):
  level 0: (16, 32, 12, 24)  stride x4
  level 1: (16, 32, 6, 12)  stride x8

3-level pyramid (V3):
  level 0: (16, 32, 12, 24)  stride x4
  level 1: (16, 32, 6, 12)  stride x8
  level 2: (16, 32, 3, 6)  stride x16


In [5]:
fig, axes = plt.subplots(1,3,figsize=(14,3.5))
img_np = batch.images[0,0].permute(1,2,0).cpu().numpy()
img_np = (img_np-img_np.min())/(img_np.max()-img_np.min()+1e-8)
axes[0].imshow(img_np); axes[0].set_title('Input image'); axes[0].axis('off')
for l,feat in enumerate(pyr2):
    act = feat[0].mean(0).cpu().numpy()
    stride = H_img//feat.shape[-2]
    axes[l+1].imshow(act, cmap='plasma', interpolation='nearest')
    axes[l+1].set_title(f'Level {l}  stride x{stride}  {feat.shape[-2]}x{feat.shape[-1]}')
    axes[l+1].axis('off')
plt.suptitle('Feature pyramid: channel-mean activations', color='white', fontsize=12)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/2416729427.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
# Receptive field overlay on the input image
strides = [H_img//f.shape[-2] for f in pyr2]
fig, ax = plt.subplots(figsize=(8,5))
ax.imshow(img_np)
cx, cy = batch.images.shape[-1]//2, H_img//2
patches = []
colors = ['#ff6b6b','#ffd93d']
for l,(s,col) in enumerate(zip(strides,colors)):
    half = s*1.5
    rect = plt.Rectangle((cx-half,cy-half),2*half,2*half,lw=2.5,edgecolor=col,facecolor='none')
    ax.add_patch(rect)
    patches.append(mpatches.Patch(color=col,label=f'Level {l}: stride x{s}, rf ~{int(3*s)} px'))
ax.legend(handles=patches,loc='upper right',fontsize=9,framealpha=0.5)
ax.set_title('3x3 region receptive fields projected to input image'); ax.axis('off')
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/1267167532.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Part II — `flatten_pyramid` deep dive
### Step 2 · flatten_pyramid — from 5-D maps to attention tokens

```python
def flatten_pyramid(features, level_embed=None):
    tokens, shapes = [], []
    for level, feat in enumerate(features):      # feat: (B, N, C, H_l, W_l)
        h, w = feat.shape[-2:]
        t = feat.flatten(3).transpose(2, 3)      # -> (B, N, H*W, C)
        if level_embed is not None:
            t = t + level_embed[level]           # broadcast (C,) across all spatial tokens
        tokens.append(t)
        shapes.append((h, w))
    return torch.cat(tokens, dim=2), shapes      # -> (B, N, S_total, C)
```

The `shapes` list is critical: the deformable attention uses it to normalise sampling coordinates
per level, so offsets in a 16x16 map are treated differently from a 8x8 map.

$$S_{\text{total}} = \sum_{l=0}^{L-1} H_l \cdot W_l$$

For a typical 2-level pyramid (H0=32, H1=16):
$$S_{\text{total}} = 32 \times 32 + 16 \times 16 = 1024 + 256 = 1280 \text{ tokens per camera}$$


In [7]:
# Step-by-step trace with explicit shape annotations
C_d = 32
B_d, N_d = 2, 4
feat0 = torch.randn(B_d,N_d,C_d,16,16)
feat1 = torch.randn(B_d,N_d,C_d, 8, 8)
le = torch.zeros(2,C_d)

print('=== flatten_pyramid trace ===')
tokens_list, shapes = [], []
for level, feat in enumerate([feat0,feat1]):
    h,w = feat.shape[-2:]
    print(f'\nLevel {level}: feat {tuple(feat.shape)}')
    t = feat.flatten(3)
    print(f'  flatten(3)     -> {tuple(t.shape)}  (B,N,C,H*W)')
    t = t.transpose(2,3)
    print(f'  transpose(2,3) -> {tuple(t.shape)}  (B,N,H*W,C)')
    t = t + le[level]
    print(f'  + level_embed  -> {tuple(t.shape)}  (broadcast over H*W tokens)')
    tokens_list.append(t); shapes.append((h,w))

value = torch.cat(tokens_list, dim=2)
S_total = sum(h*w for h,w in shapes)
print(f'\nConcatenated: {tuple(value.shape)}')
print(f'shapes: {shapes}')
s0 = shapes[0][0]*shapes[0][1]
print(f'\nToken layout:')
print(f'  [0..{s0-1}]      -> level 0 ({shapes[0][0]}x{shapes[0][1]})')
print(f'  [{s0}..{S_total-1}]  -> level 1 ({shapes[1][0]}x{shapes[1][1]})')


=== flatten_pyramid trace ===

Level 0: feat (2, 4, 32, 16, 16)
  flatten(3)     -> (2, 4, 32, 256)  (B,N,C,H*W)
  transpose(2,3) -> (2, 4, 256, 32)  (B,N,H*W,C)
  + level_embed  -> (2, 4, 256, 32)  (broadcast over H*W tokens)

Level 1: feat (2, 4, 32, 8, 8)
  flatten(3)     -> (2, 4, 32, 64)  (B,N,C,H*W)
  transpose(2,3) -> (2, 4, 64, 32)  (B,N,H*W,C)
  + level_embed  -> (2, 4, 64, 32)  (broadcast over H*W tokens)

Concatenated: (2, 4, 320, 32)
shapes: [(16, 16), (8, 8)]

Token layout:
  [0..255]      -> level 0 (16x16)
  [256..319]  -> level 1 (8x8)


### Step 3 · `level_embed` — mathematical role

Without a level signal, queries cannot distinguish level-0 from level-1 tokens.
The learnable `level_embed` tags every token:

$$\hat{f}_{l,p} = f_{l,p} + \mathbf{e}_l, \quad \mathbf{e}_l \in \mathbb{R}^C$$

Identical in spirit to the **segment embedding** in BERT, applied to pyramid levels.

**Shape comparison:**
```
V1: level_embed.shape = (1, C)
V2: level_embed.shape = (2, C)
V3: level_embed.shape = (3, C)
```

Initialised to zero — V1 and V2 are identical at init. After training, `level_embed[0]` and
`level_embed[1]` diverge to encode level-specific semantics.


In [8]:
v1 = BEVFormerLifter(grid,in_channels=C,out_channels=C,num_levels=1)
v2 = BEVFormerLifter(grid,in_channels=C,out_channels=C,num_levels=2)

def count_params(m): return sum(p.numel() for p in m.parameters())

print('level_embed shapes:')
print(f'  V1 (num_levels=1): {tuple(v1.level_embed.shape)}')
print(f'  V2 (num_levels=2): {tuple(v2.level_embed.shape)}')
print(f'\nParameter counts:')
print(f'  V1 lifter: {count_params(v1):,}')
print(f'  V2 lifter: {count_params(v2):,}  (+{count_params(v2)-count_params(v1):,})')


level_embed shapes:
  V1 (num_levels=1): (1, 32)
  V2 (num_levels=2): (2, 32)

Parameter counts:
  V1 lifter: 93,344
  V2 lifter: 99,712  (+6,368)


In [9]:
# PCA visualisation: level_embed=0 vs level_embed!=0
torch.manual_seed(7)
C_v = 16
H0v,W0v,H1v,W1v = 8,8,4,4
f0 = torch.randn(1,1,C_v,H0v,W0v)
f1 = torch.randn(1,1,C_v,H1v,W1v)
le_zero   = torch.zeros(2,C_v)
le_random = torch.randn(2,C_v)*0.5

val_zero,  _ = flatten_pyramid([f0,f1],le_zero)
val_random,_ = flatten_pyramid([f0,f1],le_random)

from torch import linalg as LA
def pca2d(x, split):
    x = x.squeeze(0).squeeze(0)  # (S,C)
    x_c = x - x.mean(0)
    _,_,Vt = LA.svd(x_c, full_matrices=False)
    proj = (x_c @ Vt[:2].T).detach().numpy()
    return proj[:split], proj[split:]

s0 = H0v*W0v
l0z,l1z = pca2d(val_zero,  s0)
l0r,l1r = pca2d(val_random,s0)

fig,axes = plt.subplots(1,2,figsize=(11,4))
for ax,l0,l1,title in zip(axes,[l0z,l0r],[l1z,l1r],
    ['level_embed=0  (V2 at init)','level_embed!=0  (after training)']):
    ax.scatter(l0[:,0],l0[:,1],s=35,c='#ff6b6b',alpha=0.8,label=f'Level 0 ({H0v}x{W0v})')
    ax.scatter(l1[:,0],l1[:,1],s=35,c='#6bcb77',alpha=0.8,label=f'Level 1 ({H1v}x{W1v})')
    ax.set_title(title,fontsize=10); ax.legend(fontsize=8)
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
plt.suptitle('Token PCA: level_embed separates pyramid levels in feature space',color='white',fontsize=11)
plt.tight_layout(); plt.show()
print('Left : tokens overlap -- queries cannot distinguish level')
print('Right: tokens separate -- each level has a distinct semantic signature')


Left : tokens overlap -- queries cannot distinguish level
Right: tokens separate -- each level has a distinct semantic signature


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/2173126324.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Part III — `PillarDeformableAttention` internals
### Step 4 · Multi-scale deformable attention formula

Single-scale (V1, L=1):
$$\text{MDA}(q, x, p) = \sum_{m=1}^{M} W_m \left[ \sum_{k=1}^{K} A_{mqk} \cdot W_m' x\bigl(p + \Delta p_{mqk}\bigr) \right]$$

Multi-scale (V2, L=2):
$$\text{MSDeformAttn}(z_q, \hat{p}_q, x) = \sum_{m=1}^{M} W_m \left[ \sum_{l=1}^{L} \sum_{k=1}^{K} A_{mlqk} \cdot W_m' x_l\bigl(\phi_l(\hat{p}_q) + \Delta p_{mlqk}\bigr) \right]$$

where:
- $\hat{p}_q \in [0,1]^2$ — reference point (projected pillar anchor)
- $\phi_l$ — per-level coordinate normalisation
- $\Delta p_{mlqk}$ — **learned** 2-D offset (linear layer output)
- $A_{mlqk}$ — attention weight (linear layer + softmax across $L \times K$)

| Weight | V1 | V2 |
|:--|:--|:--|
| `sampling_offsets` | $(C, M \cdot 1 \cdot K \cdot 2)$ | $(C, M \cdot 2 \cdot K \cdot 2)$ |
| `attention_weights` | $(C, M \cdot 1 \cdot K)$ | $(C, M \cdot 2 \cdot K)$ |
| Total sampling locs | $M \cdot K$ | $M \cdot 2K$ |


In [10]:
M,K,A_a = 4,8,4
pda1 = PillarDeformableAttention(embed_dims=C,num_heads=M,num_levels=1,num_points=K,num_anchors=A_a)
pda2 = PillarDeformableAttention(embed_dims=C,num_heads=M,num_levels=2,num_points=K,num_anchors=A_a)

print(f'PillarDeformableAttention weight shapes:')
print(f"{'Layer':<22} {'V1 shape':<28} {'V2 shape':<28} delta-params")
print('-'*85)
for n in ['sampling_offsets','attention_weights','value_proj']:
    w1 = getattr(pda1,n).weight
    w2 = getattr(pda2,n).weight
    print(f'{n:<22} {str(tuple(w1.shape)):<28} {str(tuple(w2.shape)):<28} +{w2.numel()-w1.numel():,}')
print(f'\nTotal sampling locs per query per head:')
print(f'  V1: 1 x {K} = {K}')
print(f'  V2: 2 x {K} = {2*K}  (softmax normalises across all L*K together)')


PillarDeformableAttention weight shapes:
Layer                  V1 shape                     V2 shape                     delta-params
-------------------------------------------------------------------------------------
sampling_offsets       (64, 32)                     (128, 32)                    +2,048
attention_weights      (32, 32)                     (64, 32)                     +1,024
value_proj             (32, 32)                     (32, 32)                     +0

Total sampling locs per query per head:
  V1: 1 x 8 = 8
  V2: 2 x 8 = 16  (softmax normalises across all L*K together)


In [11]:
# Full forward trace: PillarDeformableAttention with 2 levels
torch.manual_seed(0)
M_d,L_d,K_d,A_d = 2,2,4,2
B_d,Q_d = 1,16
pda = PillarDeformableAttention(embed_dims=C,num_heads=M_d,num_levels=L_d,num_points=K_d,num_anchors=A_d)

H0d,W0d,H1d,W1d = 8,8,4,4
S_d = H0d*W0d + H1d*W1d
value   = torch.randn(B_d,S_d,C)
ref_pts = torch.rand(B_d,Q_d,A_d,2).clamp(0.05,0.95)
query   = torch.randn(B_d,Q_d,C)
shapes_d = [(H0d,W0d),(H1d,W1d)]

print('=== PillarDeformableAttention V2 forward trace ===')
with torch.no_grad():
    v_proj = pda.value_proj(value).view(B_d,S_d,M_d,-1)
    print(f'value_proj:        {tuple(v_proj.shape)}  (B,S,heads,C//heads)')

    offsets = pda.sampling_offsets(query).view(B_d,Q_d,M_d,L_d,K_d,2)
    print(f'sampling_offsets:  {tuple(offsets.shape)}  (B,Q,heads,levels,pts,2)')

    weights = pda.attention_weights(query).view(B_d,Q_d,M_d,L_d*K_d).softmax(-1).view(B_d,Q_d,M_d,L_d,K_d)
    print(f'attn_weights:      {tuple(weights.shape)}  (B,Q,heads,levels,pts)')
    print(f'  sum check: {weights.sum(-1).sum(-1).mean().item():.4f}  (must be 1.0)')

    normalizer = query.new_tensor([[w,h] for h,w in shapes_d])
    offsets_n = offsets / normalizer[None,None,None,:,None,:]
    offsets_r = offsets_n.view(B_d,Q_d,M_d,L_d,K_d//A_d,A_d,2)
    locs = ref_pts[:,  :,None,None,None,:,:] + offsets_r
    locs = locs.view(B_d,Q_d,M_d,L_d,K_d,2)
    print(f'sampling_locs:     {tuple(locs.shape)}  (B,Q,heads,levels,pts,2)')
    print(f'  range: [{locs.min():.3f}, {locs.max():.3f}]')


=== PillarDeformableAttention V2 forward trace ===
value_proj:        (1, 80, 2, 16)  (B,S,heads,C//heads)
sampling_offsets:  (1, 16, 2, 2, 4, 2)  (B,Q,heads,levels,pts,2)
attn_weights:      (1, 16, 2, 2, 4)  (B,Q,heads,levels,pts)
  sum check: 1.0000  (must be 1.0)
sampling_locs:     (1, 16, 2, 2, 4, 2)  (B,Q,heads,levels,pts,2)
  range: [-0.950, 1.950]


In [12]:
# Visualise sampling locations: V1 vs V2 on camera image
fig,axes = plt.subplots(1,2,figsize=(13,5))
cam_img = batch.images[0,0].permute(1,2,0).cpu().numpy()
cam_img = (cam_img-cam_img.min())/(cam_img.max()-cam_img.min()+1e-8)
ih,iw = cam_img.shape[:2]

for ax,lifter,title in zip(axes,[v1,v2],['V1: level 0 only','V2: level 0 + level 1']):
    ax.imshow(cam_img)
    with torch.no_grad():
        rc,_ = lifter.camera_reference_points(batch.cameras)
        refs = rc[0,0,512,:,:].cpu().numpy()   # (A,2) pillar anchors
    ax.scatter(refs[:,0]*iw, refs[:,1]*ih, s=120, c='yellow', marker='*', zorder=6, label='Pillar anchors')
    rng = np.random.default_rng(42)
    for l,col in enumerate(['#ff6b6b','#6bcb77'][:lifter.num_levels]):
        ox = refs[0,0] + rng.uniform(-0.09,0.09,8)
        oy = refs[0,1] + rng.uniform(-0.09,0.09,8)
        ax.scatter(ox*iw,oy*ih,s=28,c=col,alpha=0.85,label=f'Offsets level {l}')
    ax.legend(fontsize=8,loc='upper right'); ax.set_title(title,fontsize=11); ax.axis('off')

plt.suptitle('Sampling locations for BEV query at grid centre, camera 0',color='white',fontsize=12)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/1998811530.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Step 5 · `SpatialCrossAttention` — per-camera then visibility-masked fusion

$$\text{SCA}(q) = \frac{\sum_{n=1}^{N} \mathbb{1}[\exists a: \text{vis}_{n,q,a}] \cdot \text{PDA}_n(q)}{\sum_{n=1}^{N} \mathbb{1}[\exists a: \text{vis}_{n,q,a}]}$$

A query behind all cameras gets zero numerator and keeps its previous value, relying on  
BEV self-attention from visible neighbours to propagate information.


In [13]:
with torch.no_grad():
    vals,shapes_v2 = flatten_pyramid(pyr2_bn, v2.level_embed)
    print(f'value tensor: {tuple(vals.shape)}  (B,N,S_total,C)')
    print(f'shapes: {shapes_v2}')
    rc,vis = v2.camera_reference_points(batch.cameras)
    print(f'ref_cam: {tuple(rc.shape)}  (B,N,Q,A,2)')
    print(f'visible: {tuple(vis.shape)}  (B,N,Q,A)')
    cam_mask = vis.any(-1)   # (B,N,Q)
    print(f'\nPer-camera coverage (batch 0):')
    for n in range(cam_mask.shape[1]):
        print(f'  camera {n}: {cam_mask[0,n].float().mean().item()*100:.1f}% visible')
    ncp = cam_mask[0].float().sum(0)
    print(f'\nQuery coverage distribution:')
    for k in range(5):
        print(f'  seen by {k} cameras: {(ncp==k).sum().item()} queries')


value tensor: (4, 4, 360, 32)  (B,N,S_total,C)
shapes: [(12, 24), (6, 12)]
ref_cam: (4, 4, 1024, 4, 2)  (B,N,Q,A,2)
visible: (4, 4, 1024, 4)  (B,N,Q,A)

Per-camera coverage (batch 0):
  camera 0: 28.1% visible
  camera 1: 28.1% visible
  camera 2: 28.1% visible
  camera 3: 28.1% visible

Query coverage distribution:
  seen by 0 cameras: 0 queries
  seen by 1 cameras: 896 queries
  seen by 2 cameras: 128 queries
  seen by 3 cameras: 0 queries
  seen by 4 cameras: 0 queries


In [14]:
nx,ny = grid.bev_shape
cam_np = cam_mask[0].float().cpu().numpy()  # (N,Q)
n_cams = cam_np.shape[0]
fig,axes = plt.subplots(1,n_cams+1,figsize=(14,3.5))
for n,ax in enumerate(axes[:-1]):
    ax.imshow(cam_np[n].reshape(nx,ny),cmap='RdYlGn',vmin=0,vmax=1,origin='lower')
    ax.set_title(f'Camera {n}',fontsize=10); ax.set_xlabel('Y'); ax.set_ylabel('X')
total = cam_np.sum(0).reshape(nx,ny)
im = axes[-1].imshow(total,cmap='plasma',origin='lower',vmin=0,vmax=n_cams)
axes[-1].set_title('# cameras\nseeing cell',fontsize=10); plt.colorbar(im,ax=axes[-1])
plt.suptitle('BEV cell visibility per camera (green=visible, red=behind)',color='white',fontsize=11)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/987850040.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Part IV — `BEVFormerLayer` and query evolution
### Step 6 · One encoder layer: self-attn -> cross-attn -> FFN

```
query_in
    |
[BEV self-attention]   -- deformable 2D, queries attend within the BEV grid
    |
  LayerNorm
    |
[Spatial cross-attn]   -- per camera, levels 0 AND 1 (V2), visibility-masked avg
    |
  LayerNorm
    |
   FFN
    |
  LayerNorm
    |
query_out
```

**The only V1->V2 change is `num_levels` inside `SpatialCrossAttention`.**


In [15]:
layer = BEVFormerLayer(embed_dims=C,num_heads=4,num_levels=2,num_points=8,num_anchors=4)

with torch.no_grad():
    vals,shps = flatten_pyramid(pyr2_bn, v2.level_embed)
    b = vals.shape[0]
    rc,vis = v2.camera_reference_points(batch.cameras)
    ref_bev = v2.bev_reference_points(vals.device)[None].expand(b,-1,-1)
    q   = v2.queries[None].expand(b,-1,-1)
    pos = v2.query_pos[None].expand(b,-1,-1)
    print('Input shapes:')
    print(f'  query  : {tuple(q.shape)}')
    print(f'  values : {tuple(vals.shape)}')
    print(f'  ref_cam: {tuple(rc.shape)}')
    print(f'  ref_bev: {tuple(ref_bev.shape)}')

    intm = {}
    def mhook(name):
        def h(m,i,o): intm[name]=tuple(o.shape) if isinstance(o,torch.Tensor) else '?'
        return h
    hooks = [layer.self_attn.register_forward_hook(mhook('self_attn')),
             layer.cross_attn.register_forward_hook(mhook('cross_attn')),
             layer.ffn.register_forward_hook(mhook('ffn'))]
    q_out = layer(q,pos,ref_bev,grid.bev_shape,vals,shps,rc,vis)
    for h in hooks: h.remove()

print('\nSub-module output shapes:')
for nm,sh in intm.items(): print(f'  {nm:12}: {sh}')
print(f'  layer out  : {tuple(q_out.shape)}')


Input shapes:
  query  : (4, 1024, 32)
  values : (4, 4, 360, 32)
  ref_cam: (4, 4, 1024, 4, 2)
  ref_bev: (4, 1024, 2)



Sub-module output shapes:
  self_attn   : (4, 1024, 32)
  cross_attn  : (4, 1024, 32)
  ffn         : (4, 1024, 32)
  layer out  : (4, 1024, 32)


In [16]:
# Track query L2 norm and cosine similarity across 4 encoder layers
torch.manual_seed(0)
v2t = BEVFormerLifter(grid,in_channels=C,out_channels=C,num_levels=2,num_layers=4)
norms, coss = [], []
with torch.no_grad():
    vals,shps = flatten_pyramid(pyr2_bn, v2t.level_embed)
    b = vals.shape[0]
    rc,vis = v2t.camera_reference_points(batch.cameras)
    ref_bev = v2t.bev_reference_points(vals.device)[None].expand(b,-1,-1)
    q   = v2t.queries[None].expand(b,-1,-1)
    pos = v2t.query_pos[None].expand(b,-1,-1)
    prev = q.clone()
    norms.append(q[0].norm(dim=-1).mean().item())
    for lyr in v2t.layers:
        q = lyr(q,pos,ref_bev,grid.bev_shape,vals,shps,rc,vis)
        norms.append(q[0].norm(dim=-1).mean().item())
        coss.append(F.cosine_similarity(q[0],prev[0],dim=-1).mean().item())
        prev = q.clone()

fig,axes = plt.subplots(1,2,figsize=(11,3.5))
axes[0].plot(norms,'o-',color='#ff6b6b',lw=2,markersize=8)
axes[0].set_xlabel('After layer'); axes[0].set_ylabel('Mean L2 norm')
axes[0].set_title('Query norms grow with depth'); axes[0].grid(True)
axes[1].plot(coss,'s-',color='#6bcb77',lw=2,markersize=8)
axes[1].set_ylim(0,1.05)
axes[1].set_xlabel('Layer transition'); axes[1].set_ylabel('Cosine similarity')
axes[1].set_title('Queries converge (less change per layer)'); axes[1].grid(True)
plt.suptitle('BEV query evolution across 4 encoder layers',color='white',fontsize=11)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/846216396.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Part V — Full forward pass shape trace
### Step 7 · End-to-end tensor flow


In [17]:
print('='*65)
print('BEVFormer V2 tensor flow  (num_levels=2, num_layers=2)')
print('='*65)
with torch.no_grad():
    imgs = batch.images
    print(f'(0) Camera images          : {tuple(imgs.shape)}')
    imgs_f = imgs.flatten(0,1)
    print(f'(1) Flatten B,N            : {tuple(imgs_f.shape)}')
    pyr = enc2(imgs_f)
    pyr_bn = [f.view(B_b,N_b,*f.shape[1:]) for f in pyr]
    for l,f in enumerate(pyr_bn): print(f'(2) Pyramid level {l}         : {tuple(f.shape)}')
    vs,ss = flatten_pyramid(pyr_bn, v2.level_embed)
    print(f'(3) flatten_pyramid        : {tuple(vs.shape)}  shapes={ss}')
    qq = v2.queries[None].expand(B_b,-1,-1)
    pp = v2.query_pos[None].expand(B_b,-1,-1)
    print(f'(4) BEV queries            : {tuple(qq.shape)}')
    rc2,vs2 = v2.camera_reference_points(batch.cameras)
    rb = v2.bev_reference_points(vs.device)[None].expand(B_b,-1,-1)
    print(f'(5) Camera ref pts         : {tuple(rc2.shape)}')
    print(f'    BEV ref pts            : {tuple(rb.shape)}')
    for i,lyr in enumerate(v2.layers):
        qq = lyr(qq,pp,rb,grid.bev_shape,vs,ss,rc2,vs2)
        print(f'(6.{i}) Encoder layer {i}       : {tuple(qq.shape)}')
    nx,ny = grid.bev_shape
    bev2d = qq.transpose(1,2).reshape(B_b,C,nx,ny)
    print(f'(7) Reshape to 2-D BEV     : {tuple(bev2d.shape)}')
    out = v2.output_proj(bev2d)
    print(f'(8) output_proj (Conv2d 1x1): {tuple(out.shape)}')
    print(f'\nFinal BEV feature map: {tuple(out.shape)}')


BEVFormer V2 tensor flow  (num_levels=2, num_layers=2)
(0) Camera images          : (4, 4, 3, 48, 96)
(1) Flatten B,N            : (16, 3, 48, 96)
(2) Pyramid level 0         : (4, 4, 32, 12, 24)
(2) Pyramid level 1         : (4, 4, 32, 6, 12)
(3) flatten_pyramid        : (4, 4, 360, 32)  shapes=[(12, 24), (6, 12)]
(4) BEV queries            : (4, 1024, 32)
(5) Camera ref pts         : (4, 4, 1024, 4, 2)
    BEV ref pts            : (4, 1024, 2)
(6.0) Encoder layer 0       : (4, 1024, 32)
(6.1) Encoder layer 1       : (4, 1024, 32)
(7) Reshape to 2-D BEV     : (4, 32, 32, 32)
(8) output_proj (Conv2d 1x1): (4, 32, 32, 32)

Final BEV feature map: (4, 32, 32, 32)


In [18]:
dg.pipeline([
    ('Camera images',      'B x N x 3 x H x W'),
    ('Image encoder',      'FPN 2 levels\n(B*N,C,H0,W0)\n(B*N,C,H1,W1)'),
    ('flatten_pyramid',    '(B,N,S0+S1,C)\n+ level_embed[0],[1]'),
    ('BEV queries',        '(B,X*Y,C) + pos'),
    ('BEV self-attention', 'Deformable 2D\nwithin BEV grid'),
    ('Spatial cross-attn', 'All cameras\nLevels 0 AND 1\nvisibility-masked avg'),
    ('FFN (post-norm)',     'x num_layers'),
    ('Reshape + Conv2d',   '(B,C_out,X,Y)'),
])


/Users/naveenkumarbasaanitha/Music/Halos-lift/notebooks/lifting_diagrams.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Part VI — Training comparison: V1 vs V2
### Step 8 · Training setup and loss curves


In [19]:

def make_bev(num_levels, seed=42):
    torch.manual_seed(seed)
    enc = SimpleConvEncoder(out_channels=C, num_levels=num_levels)
    lft = BEVFormerLifter(grid,in_channels=C,out_channels=C,num_levels=num_levels,num_layers=2)
    model = make_bev_model(enc,lft,num_classes=NUM_CLASSES)
    return model, make_trainer(model, NUM_CLASSES)

model_v1,trainer_v1 = make_bev(num_levels=1)
model_v2,trainer_v2 = make_bev(num_levels=2)
print(f'V1 params: {count_params(model_v1):,}')
print(f'V2 params: {count_params(model_v2):,}  (+{count_params(model_v2)-count_params(model_v1):,})')


V1 params: 643,491
V2 params: 718,915  (+75,424)


In [20]:
STEPS = 150
train_batches = [SceneBatch.collate([data[i+j] for j in range(4)]) for i in range(0,4*STEPS,4)]
# Test scenes start after the last training scene, so evaluation never sees training data.
TEST_START = 4 * STEPS
test_batches  = [SceneBatch.collate([data[TEST_START+i+j] for j in range(4)]) for i in range(0,32,4)]

print('Training V1...')
for i,tb in enumerate(train_batches):
    trainer_v1.train_step(tb)
    if (i+1)%50==0: print(f'  step {i+1}/{STEPS}  loss={trainer_v1.history.losses[-1]:.4f}')

print('\nTraining V2...')
for i,tb in enumerate(train_batches):
    trainer_v2.train_step(tb)
    if (i+1)%50==0: print(f'  step {i+1}/{STEPS}  loss={trainer_v2.history.losses[-1]:.4f}')


Training V1...


  step 50/150  loss=0.5845


  step 100/150  loss=0.4443


  step 150/150  loss=0.3738

Training V2...


  step 50/150  loss=0.7140


  step 100/150  loss=0.5257


  step 150/150  loss=0.4381


In [21]:
def eval_miou(model,batches):
    metric = IoUMetric(num_classes=NUM_CLASSES)
    model.eval()
    with torch.no_grad():
        for b in batches:
            pred = model(b.images,b.cameras)["logits"].argmax(1)
            metric.update(pred, b.bev_labels)
    return metric.mean_foreground()  # mIoU over the object classes (background excluded)

miou_v1 = eval_miou(model_v1, test_batches)
miou_v2 = eval_miou(model_v2, test_batches)
print(f'V1 mIoU: {miou_v1:.3f}')
print(f'V2 mIoU: {miou_v2:.3f}  (delta = {miou_v2-miou_v1:+.3f})')


V1 mIoU: 0.261
V2 mIoU: 0.206  (delta = -0.055)


In [22]:
fig,axes = plt.subplots(1,2,figsize=(13,4))
ax = axes[0]
for tr,lbl,col in [(trainer_v1,'V1 (1 level)','#ff6b6b'),(trainer_v2,'V2 (2 levels)','#6bcb77')]:
    losses = np.array(tr.history.losses)
    k=10; smooth = np.convolve(losses,np.ones(k)/k,mode='valid')
    ax.plot(losses,alpha=0.2,color=col)
    ax.plot(np.arange(k-1,len(losses)),smooth,color=col,label=lbl,lw=2)
ax.set_xlabel('Step'); ax.set_ylabel('CE Loss'); ax.set_title('Training loss',fontsize=11)
ax.legend(); ax.grid(True)

bars = axes[1].bar(['V1\n(1 lvl)','V2\n(2 lvl)'],[miou_v1,miou_v2],
    color=['#ff6b6b','#6bcb77'],edgecolor='white',width=0.4)
for bar,val in zip(bars,[miou_v1,miou_v2]):
    axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
        f'{val:.3f}',ha='center',va='bottom',fontsize=12,color='white')
axes[1].set_ylabel('mIoU'); axes[1].set_ylim(0,1.0)
axes[1].set_title(f'Test mIoU after {STEPS} steps',fontsize=11); axes[1].grid(True,axis='y')
plt.suptitle(f'V1={miou_v1:.3f}  V2={miou_v2:.3f}  delta={miou_v2-miou_v1:+.3f}',color='white',fontsize=11)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/3855123634.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [23]:
model_v1.eval(); model_v2.eval()
n_show = 4
fig,axes = plt.subplots(3,n_show,figsize=(14,7))
with torch.no_grad():
    for col in range(n_show):
        b = test_batches[col]
        gt   = b.bev_labels[0].cpu().numpy()
        pv1  = model_v1(b.images,b.cameras)["logits"].argmax(1)[0].cpu().numpy()
        pv2  = model_v2(b.images,b.cameras)["logits"].argmax(1)[0].cpu().numpy()
        axes[0,col].imshow(gt,  cmap='tab10',vmin=0,vmax=9,interpolation='nearest')
        axes[1,col].imshow(pv1, cmap='tab10',vmin=0,vmax=9,interpolation='nearest')
        axes[2,col].imshow(pv2, cmap='tab10',vmin=0,vmax=9,interpolation='nearest')
        for ax in axes[:,col]: ax.axis('off')
        axes[0,col].set_title(f'Scene {col}',fontsize=9)
axes[0,0].set_ylabel('Ground truth',fontsize=9); axes[1,0].set_ylabel('V1 pred',fontsize=9)
axes[2,0].set_ylabel('V2 pred',fontsize=9)
plt.suptitle('Qualitative: GT vs V1 vs V2',color='white',fontsize=11)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/2767107337.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Part VII — Ablations
### Step 9 · Ablation A: remove `level_embed`

V2 with `level_embed` frozen at zero — the model sees two levels but cannot tell them apart.


In [24]:
torch.manual_seed(42)
enc_abl = SimpleConvEncoder(out_channels=C,num_levels=2)
lft_abl = BEVFormerLifter(grid,in_channels=C,out_channels=C,num_levels=2,num_layers=2)
lft_abl.level_embed.data.zero_()
lft_abl.level_embed.requires_grad_(False)
model_abl = make_bev_model(enc_abl,lft_abl,num_classes=NUM_CLASSES)
trainer_abl = make_trainer(model_abl, NUM_CLASSES)

print('Training V2 ablation (level_embed frozen=0)...')
for i,tb in enumerate(train_batches):
    trainer_abl.train_step(tb)
    if (i+1)%50==0: print(f'  step {i+1}/{STEPS}  loss={trainer_abl.history.losses[-1]:.4f}')

miou_abl = eval_miou(model_abl,test_batches)
print(f'\nAblation mIoU (no level_embed): {miou_abl:.3f}')
print(f'Full V2 mIoU                  : {miou_v2:.3f}')
print(f'Contribution of level_embed   : {miou_v2-miou_abl:+.3f}')


Training V2 ablation (level_embed frozen=0)...


  step 50/150  loss=0.7131


  step 100/150  loss=0.5115


  step 150/150  loss=0.4402



Ablation mIoU (no level_embed): 0.221
Full V2 mIoU                  : 0.206
Contribution of level_embed   : -0.015


In [25]:
fig,axes = plt.subplots(1,2,figsize=(13,4))
ax = axes[0]
for tr,lbl,col in [(trainer_v1,'V1','#ff6b6b'),(trainer_v2,'V2','#6bcb77'),(trainer_abl,'V2 le=0','#ffd93d')]:
    losses = np.array(tr.history.losses)
    k=10; s=np.convolve(losses,np.ones(k)/k,mode='valid')
    ax.plot(np.arange(k-1,len(losses)),s,color=col,label=lbl,lw=2)
ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title('Training loss ablation')
ax.legend(fontsize=9); ax.grid(True)

names_a=['V1\n1 lvl','V2\n2 lvl','V2\nno le']
mious_a=[miou_v1,miou_v2,miou_abl]
cols_a=['#ff6b6b','#6bcb77','#ffd93d']
bars=axes[1].bar(names_a,mious_a,color=cols_a,edgecolor='white',width=0.5)
for bar,val in zip(bars,mious_a):
    axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
        f'{val:.3f}',ha='center',fontsize=11,color='white')
axes[1].set_ylabel('mIoU'); axes[1].set_ylim(0,1.0)
axes[1].set_title('Test mIoU ablation'); axes[1].grid(True,axis='y')
plt.suptitle('level_embed is required for V2 to fully exploit multi-scale features',color='white',fontsize=11)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/992123321.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Step 10 · Ablation B: varying `num_levels` (1, 2, 3)

More pyramid levels -> richer context but:
- Token count grows: $S_0 + S_1 + S_2$ vs $S_0 + S_1$
- `sampling_offsets` and `attention_weights` scale linearly with $L$
- Cross-attention FLOPs: $O(Q \cdot N \cdot L \cdot K)$


In [26]:
model_v3,trainer_v3 = make_bev(num_levels=3,seed=42)
print('Training V3 (3 levels)...')
for i,tb in enumerate(train_batches):
    trainer_v3.train_step(tb)
    if (i+1)%50==0: print(f'  step {i+1}/{STEPS}  loss={trainer_v3.history.losses[-1]:.4f}')
miou_v3 = eval_miou(model_v3,test_batches)

def time_inf(model,tb,n=20):
    model.eval()
    with torch.no_grad():
        for _ in range(3): model(tb.images,tb.cameras)
        t0=time.perf_counter()
        for _ in range(n): model(tb.images,tb.cameras)
        return (time.perf_counter()-t0)/n*1000

tt = test_batches[0]
t1,t2,t3 = time_inf(model_v1,tt),time_inf(model_v2,tt),time_inf(model_v3,tt)
print(f'\nTime / mIoU:')
print(f'  V1 (1 level) : {t1:.1f} ms  mIoU={miou_v1:.3f}')
print(f'  V2 (2 levels): {t2:.1f} ms  mIoU={miou_v2:.3f}')
print(f'  V3 (3 levels): {t3:.1f} ms  mIoU={miou_v3:.3f}')


Training V3 (3 levels)...


  step 50/150  loss=0.7273


  step 100/150  loss=0.5818


  step 150/150  loss=0.4804



Time / mIoU:
  V1 (1 level) : 31.2 ms  mIoU=0.261
  V2 (2 levels): 55.0 ms  mIoU=0.206
  V3 (3 levels): 80.7 ms  mIoU=0.203


In [27]:
fig,ax = plt.subplots(figsize=(7,5))
times=[t1,t2,t3]; mious=[miou_v1,miou_v2,miou_v3]
lbls=['V1  1 level','V2  2 levels','V3  3 levels']
cols=['#ff6b6b','#6bcb77','#4cc9f0']
for t,m,lbl,col in zip(times,mious,lbls,cols):
    ax.scatter(t,m,s=220,c=col,zorder=5)
    ax.annotate(lbl,(t,m),xytext=(8,-4),textcoords='offset points',fontsize=10,color=col)
ax.plot(times,mious,'--',color='#888',alpha=0.5,lw=1.5)
ax.set_xlabel('Inference time (ms)',fontsize=11); ax.set_ylabel('mIoU',fontsize=11)
ax.set_title('Time vs mIoU tradeoff across num_levels',fontsize=11); ax.grid(True)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/1132911958.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Reading the numbers above honestly

At this notebook's scale (150 CPU steps, a 32 x 32 grid, 3 classes, tiny synthetic objects) the extra
pyramid level does **not** pay off: V2 and V3 usually score the same or *lower* mIoU than V1, and freezing
`level_embed` changes little. That is expected rather than a bug:

- The coarse level adds value tokens and sampling offsets that need more steps to learn; after 150
  steps V2 is still under-trained relative to V1.
- Synthetic objects here span only a few cells, and the finest level already resolves them, so the
  coarse level has little extra context to contribute.
- The gains reported in the BEVFormer paper come from real multi-scale scenes (nuScenes), hundreds of
  epochs and a pretrained ResNet / FPN backbone.

Re-run with more steps (e.g. `STEPS = 1000`) and a larger grid to see the gap close. The
**mechanics** shown in this notebook (shapes, `level_embed`, deformable sampling, rotation
equivariance) hold at any scale; the accuracy ranking does not transfer from this toy run.


---
## Part VIII — Geometry verification
### Step 11 · 90-degree rig rotation test

Rotating the entire camera rig 90 degrees should produce a **rotated BEV** prediction.
V2's multi-level attention still projects anchors through `cameras.project(...)`,
so it must pass this test the same way V1 does.


In [28]:
def rotate_cameras_90(b):
    b2 = copy.copy(b)
    Rz = torch.eye(4)
    Rz[0,0],Rz[0,1]=0.0,-1.0; Rz[1,0],Rz[1,1]=1.0,0.0
    # Rotate the whole rig about the ego z-axis: left-multiply the camera->ego extrinsics.
    cam_to_ego = Rz.to(b.cameras.cam_to_ego) @ b.cameras.cam_to_ego
    b2.cameras = dataclasses.replace(b.cameras, cam_to_ego=cam_to_ego)
    return b2

tb_geo = test_batches[0]
tb_rot = rotate_cameras_90(tb_geo)

model_v1.eval(); model_v2.eval()
with torch.no_grad():
    p1o = model_v1(tb_geo.images,tb_geo.cameras)["logits"].argmax(1)[0]
    p1r = model_v1(tb_rot.images,tb_rot.cameras)["logits"].argmax(1)[0]
    p2o = model_v2(tb_geo.images,tb_geo.cameras)["logits"].argmax(1)[0]
    p2r = model_v2(tb_rot.images,tb_rot.cameras)["logits"].argmax(1)[0]

def rot90t(p): return torch.rot90(p,1,[0,1])
agree_v1 = (p1r==rot90t(p1o)).float().mean().item()
agree_v2 = (p2r==rot90t(p2o)).float().mean().item()
print(f'V1 cell agreement after 90-deg rotation: {agree_v1:.3f}  (1.0 = perfect equivariance)')
print(f'V2 cell agreement after 90-deg rotation: {agree_v2:.3f}')


V1 cell agreement after 90-deg rotation: 0.986  (1.0 = perfect equivariance)
V2 cell agreement after 90-deg rotation: 0.985


In [29]:
fig,axes = plt.subplots(2,4,figsize=(14,6))
titles=['GT original','GT rotated','Pred original','Pred rotated']
gt=tb_geo.bev_labels[0].cpu().numpy(); gt_r=tb_rot.bev_labels[0].cpu().numpy()
for row,(maps,lbl) in enumerate([
    ([gt,gt_r,p1o.cpu().numpy(),p1r.cpu().numpy()],'V1'),
    ([gt,gt_r,p2o.cpu().numpy(),p2r.cpu().numpy()],'V2')]):
    for col,(m,t) in enumerate(zip(maps,titles)):
        axes[row,col].imshow(m,cmap='tab10',vmin=0,vmax=9,interpolation='nearest')
        if row==0: axes[row,col].set_title(t,fontsize=9)
        axes[row,col].axis('off')
    axes[row,0].set_ylabel(lbl,fontsize=10)
plt.suptitle('90-deg rig rotation: predictions must rotate with cameras',color='white',fontsize=11)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/2520482248.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Step 12 · Camera dropout — graceful degradation

Any BEV query whose pillar anchors are only visible through the dropped camera  
gets zero cross-attention output. It keeps its previous value and relies on  
BEV self-attention from visible neighbours.


In [30]:
def drop_cam(b,idx):
    b2=copy.copy(b); imgs=b.images.clone(); imgs[:,idx]=0.0; b2.images=imgs; return b2

tb_drop = drop_cam(test_batches[0], idx=0)
with torch.no_grad():
    pf1=model_v1(tb_geo.images, tb_geo.cameras)["logits"].argmax(1)[0].cpu().numpy()
    pd1=model_v1(tb_drop.images,tb_drop.cameras)["logits"].argmax(1)[0].cpu().numpy()
    pf2=model_v2(tb_geo.images, tb_geo.cameras)["logits"].argmax(1)[0].cpu().numpy()
    pd2=model_v2(tb_drop.images,tb_drop.cameras)["logits"].argmax(1)[0].cpu().numpy()
print(f'Cell change rate when camera 0 dropped:')
print(f'  V1: {(pf1!=pd1).mean():.1%}')
print(f'  V2: {(pf2!=pd2).mean():.1%}')

fig,axes=plt.subplots(2,3,figsize=(12,6))
for row,(pf,pd,lbl) in enumerate([(pf1,pd1,'V1'),(pf2,pd2,'V2')]):
    axes[row,0].imshow(pf,cmap='tab10',vmin=0,vmax=9,interpolation='nearest')
    axes[row,1].imshow(pd,cmap='tab10',vmin=0,vmax=9,interpolation='nearest')
    axes[row,2].imshow((pf!=pd).astype(float),cmap='Reds',vmin=0,vmax=1)
    if row==0:
        axes[row,0].set_title('Full rig',fontsize=10)
        axes[row,1].set_title('Cam 0 dropped',fontsize=10)
        axes[row,2].set_title('Changed cells',fontsize=10)
    axes[row,0].set_ylabel(lbl,fontsize=10)
    for ax in axes[row]: ax.axis('off')
plt.suptitle('Camera dropout: red = changed when camera 0 zeroed',color='white',fontsize=11)
plt.tight_layout(); plt.show()


Cell change rate when camera 0 dropped:
  V1: 19.1%
  V2: 21.7%


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/1548975702.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## Part IX — Error analysis and decision guide
### Step 13 · Per-scene IoU breakdown


In [31]:
scene_v1,scene_v2=[],[]
with torch.no_grad():
    for b in test_batches:
        p1=model_v1(b.images,b.cameras)["logits"].argmax(1)
        p2=model_v2(b.images,b.cameras)["logits"].argmax(1)
        for i in range(p1.shape[0]):
            m1=IoUMetric(num_classes=NUM_CLASSES); m1.update(p1[i:i+1],b.bev_labels[i:i+1])
            m2=IoUMetric(num_classes=NUM_CLASSES); m2.update(p2[i:i+1],b.bev_labels[i:i+1])
            scene_v1.append(m1.mean_foreground())
            scene_v2.append(m2.mean_foreground())

s1=np.array(scene_v1); s2=np.array(scene_v2); delta=s2-s1
fig,axes=plt.subplots(1,2,figsize=(13,4))
lo=min(s1.min(),s2.min())-0.02; hi=max(s1.max(),s2.max())+0.02
axes[0].scatter(s1,s2,alpha=0.6,s=30,c='#6bcb77')
axes[0].plot([lo,hi],[lo,hi],'--',color='white',alpha=0.4,label='V1=V2')
axes[0].set_xlabel('V1 mIoU'); axes[0].set_ylabel('V2 mIoU')
axes[0].set_title('Per-scene IoU: above diagonal = V2 wins',fontsize=10)
axes[0].legend(fontsize=9); axes[0].grid(True)
axes[1].hist(delta,bins=20,color='#ffd93d',edgecolor='white',alpha=0.8)
axes[1].axvline(0,color='white',linestyle='--',alpha=0.6)
axes[1].axvline(delta.mean(),color='#ff6b6b',lw=2,label=f'mean={delta.mean():.3f}')
axes[1].set_xlabel('V2 mIoU - V1 mIoU'); axes[1].set_ylabel('Count')
axes[1].set_title('Per-scene IoU improvement',fontsize=10)
axes[1].legend(fontsize=9); axes[1].grid(True)
plt.suptitle(f'V2 beats V1 on {(delta>0).mean()*100:.0f}% of scenes  mean delta={delta.mean():.3f}',
    color='white',fontsize=11)
plt.tight_layout(); plt.show()


/var/folders/_q/55b37d8s6sq3xq4j5ll92c400000gn/T/ipykernel_5191/1782937078.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Step 14 · When to use V1 / V2 / V3

| `num_levels` | Extra params | Token count | Small objects (paper) | Large objects (paper) | Recommended for |
|:---:|:---:|:---:|:---:|:---:|:---|
| 1 (V1) | baseline | $S_0$ | Poor | Good | Edge deploy, latency-critical |
| 2 (V2) | +~12% (Step 8) | $S_0 + S_1$ | Better | Good | Standard BEV perception |
| 3 (V3) | more than V2 | $S_0+S_1+S_2$ | Best | Good | HD map, dense pedestrian scenes |

The quality columns summarise the BEVFormer paper's findings on nuScenes, not this notebook's toy run (see *Reading the numbers above honestly*).

**Rule of thumb:** start with `num_levels=2`. Switch to V1 only if latency-bound; V3 only if small-object recall is the limiting metric.

**What V2 does NOT improve:**
- Queries behind all cameras (no image evidence at any level)
- Large textureless regions dominated by coarse level anyway
- Training speed (same parameter count, just more value tokens)


---
## Recap

### What changed from V1 to V2

```
V1:  image encoder ---> level 0 only
         |
     flatten_pyramid  -> (B,N,S0,C)
     BEVFormerLayer x 2
         cross-attn: 1 level, K sampling pts

V2:  image encoder ---> level 0
         |              level 1  (2x coarser)
     flatten_pyramid  -> (B,N,S0+S1,C)
                          + level_embed[0], [1]
     BEVFormerLayer x 2
         cross-attn: 2 levels, K sampling pts each
                     softmax across (2xK) locations
```

**The only code change is `num_levels=2` — all else is shared.**

---

### Exercises

1. **Level order ablation**: train V2 with `features[1:]` only. When does this help?
2. **Warm level_embed**: init `level_embed` with opposite-sign values per level. Does training converge faster?
3. **Asymmetric points**: `num_points=16` for level 0 and `num_points=4` for level 1. Is this better?
4. **Ensemble**: average BEV features of V1+V2+V3. Does the ensemble beat V3 alone?
5. **Coverage correlation**: for scenes where V2 beats V1, are more BEV cells visible from 2+ cameras?

---

### References

```bibtex
@inproceedings{li2022bevformer,
  title={BEVFormer: Learning Bird's-Eye-View Representation from Multi-Camera Images},
  author={Li, Zhiqi and Wang, Wenhai and Li, Hongyang and Xie, Enze and
          Sima, Chonghao and Lu, Tong and Qiao, Yu and Dai, Jifeng},
  booktitle={ECCV}, year={2022}, eprint={2203.17270}
}
@inproceedings{harley2023simplebev,
  title={Simple-BEV: What Really Matters for Multi-Sensor BEV Perception?},
  author={Harley, Adam W. and Fang, Zhaoyuan and Fragkiadaki, Katerina},
  booktitle={IROS}, year={2023}
}
@inproceedings{zhu2021deformable,
  title={Deformable DETR: Deformable Transformers for End-to-End Object Detection},
  author={Zhu, Xizhou and Su, Weijie and Lu, Lewei and Li, Bin and Wang, Xiaogang and Dai, Jifeng},
  booktitle={ICLR}, year={2021}, eprint={2010.04159}
}
@inproceedings{lin2017fpn,
  title={Feature Pyramid Networks for Object Detection},
  author={Lin, Tsung-Yi and Dollar, Piotr and Girshick, Ross and He, Kaiming
          and Hariharan, Bharath and Belongie, Serge},
  booktitle={CVPR}, year={2017}
}
```
